In [ ]:
# IMPORTANT!!! ENCO only support single intervention, besides you should change ''num_categs = data_obs.max(axis=0)'' in graph_definition.py line 249

In [ ]:
import os
import sys
sys.path.append("../")
sys.path.append("../..")
sys.path.append("../../baselines/ENCO/")
sys.path.append("../../src")
import warnings
warnings.filterwarnings("ignore")
import pickle
import torch
import numpy as np
import pandas as pd
from glob import glob
from ENCO.causal_graphs.graph_definition import CausalDAGDataset
from ENCO.causal_discovery.enco import ENCO
from src.tools.metric import get_compared_components, get_skeleton, metric_skeleton_level, metric_cpdag_level

In [ ]:
class HiddenPrints:
    def __init__(self, activated=True):
        self.activated = activated
        self.original_stdout = None

    def open(self):
        sys.stdout.close()
        sys.stdout = self.original_stdout

    def close(self):
        self.original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __enter__(self):
        if self.activated:
            self.close()

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.activated:
            self.open()

In [ ]:
exp_name = 'exp_200'
intervention_size_list = [1, 1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../exps_of_result/enco/{exp_name}', exist_ok=True)

for idx, benchmark_name in enumerate(['01cancer', '02earthquake', '03survey', '04asia', '05sachs',  '06child', '07insurance', '08water', '09mildew', '10alarm', '11barley', '12hailfinder', '13hepar2', '14win95pts', '15pathfinder']):
    if idx!=4:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../exps_of_result/enco/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
        
    all_nodes_index = [all_nodes.index(item) for item in all_nodes]
    intervention_targets_index = [all_nodes.index(sublist[0]) for sublist in intervention_targets]  # only support single intervention
    
    real_exclude_inters = np.array(list(set(all_nodes_index) - set(intervention_targets_index)), dtype=np.int32)
    
    modify_data_int = np.ones((len(all_nodes), data_int.shape[1], data_int.shape[2]), dtype=np.int32)
    for i in range(data_int.shape[0]):
        modify_data_int[intervention_targets_index[i]] = data_int[i]
    
    modify_graph = CausalDAGDataset(adj_matrix=adj_matrix, data_obs=data_obs, data_int=modify_data_int, exclude_inters=real_exclude_inters)
    
    lambda_sparse = 0.02 if len(all_nodes) > 100 else 0.002
    # num_epochs = 50 if len(all_nodes) > 100 else 100
    num_epochs = 40 if len(all_nodes) > 100 else 20
    
    try:
        # with HiddenPrints():
        discovery_module = ENCO(graph=modify_graph, sample_size_obs=len(data_obs), sample_size_inters=data_int.shape[1], lambda_sparse=lambda_sparse)
        discovery_module.to(torch.device('cuda:1'))

        pred_I_CPDAG_know = discovery_module.discover_graph(num_epochs=num_epochs).detach().numpy().astype(np.uint8)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
                
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')

    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_1'
intervention_size_list = [1, 1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/enco/{exp_name}', exist_ok=True)

for idx, benchmark_name in enumerate(['01cancer', '02earthquake', '03survey', '04asia', '05sachs',  '06child', '07insurance', '08water', '09mildew', '10alarm', '11barley', '12hailfinder', '13hepar2', '14win95pts', '15pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../exps_of_result/enco/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
        
    all_nodes_index = [all_nodes.index(item) for item in all_nodes]
    intervention_targets_index = [all_nodes.index(sublist[0]) for sublist in intervention_targets]  # only support single intervention
    
    real_exclude_inters = np.array(list(set(all_nodes_index) - set(intervention_targets_index)), dtype=np.int32)
    
    modify_data_int = np.ones((len(all_nodes), data_int.shape[1], data_int.shape[2]), dtype=np.int32)
    for i in range(data_int.shape[0]):
        modify_data_int[intervention_targets_index[i]] = data_int[i]
    
    modify_graph = CausalDAGDataset(adj_matrix=adj_matrix, data_obs=data_obs, data_int=modify_data_int, exclude_inters=real_exclude_inters)
    
    lambda_sparse = 0.02 if len(all_nodes) > 100 else 0.002
    # num_epochs = 50 if len(all_nodes) > 100 else 100
    num_epochs = 40 if len(all_nodes) > 100 else 20
    
    try:
        # with HiddenPrints():
        discovery_module = ENCO(graph=modify_graph, sample_size_obs=len(data_obs), sample_size_inters=data_int.shape[1], lambda_sparse=lambda_sparse)
        discovery_module.to(torch.device('cuda:1'))

        pred_I_CPDAG_know = discovery_module.discover_graph(num_epochs=num_epochs).detach().numpy().astype(np.uint8)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
                
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')

    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_1_2'
intervention_size_list = [1, 1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/enco/{exp_name}', exist_ok=True)

for idx, benchmark_name in enumerate(['01cancer', '02earthquake', '03survey', '04asia', '05sachs',  '06child', '07insurance', '08water', '09mildew', '10alarm', '11barley', '12hailfinder', '13hepar2', '14win95pts', '15pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../exps_of_result/enco/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
        
    all_nodes_index = [all_nodes.index(item) for item in all_nodes]
    intervention_targets_index = [all_nodes.index(sublist[0]) for sublist in intervention_targets]  # only support single intervention
    
    real_exclude_inters = np.array(list(set(all_nodes_index) - set(intervention_targets_index)), dtype=np.int32)
    
    modify_data_int = np.ones((len(all_nodes), data_int.shape[1], data_int.shape[2]), dtype=np.int32)
    for i in range(data_int.shape[0]):
        modify_data_int[intervention_targets_index[i]] = data_int[i]
    
    modify_graph = CausalDAGDataset(adj_matrix=adj_matrix, data_obs=data_obs, data_int=modify_data_int, exclude_inters=real_exclude_inters)
    
    lambda_sparse = 0.02 if len(all_nodes) > 100 else 0.002
    # num_epochs = 50 if len(all_nodes) > 100 else 100
    num_epochs = 40 if len(all_nodes) > 100 else 20
    
    try:
        # with HiddenPrints():
        discovery_module = ENCO(graph=modify_graph, sample_size_obs=len(data_obs), sample_size_inters=data_int.shape[1], lambda_sparse=lambda_sparse)
        discovery_module.to(torch.device('cuda:1'))

        pred_I_CPDAG_know = discovery_module.discover_graph(num_epochs=num_epochs).detach().numpy().astype(np.uint8)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
                
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')

    except:
        print(f'pass {benchmark_name}\n')

In [ ]:
exp_name = 'exp_2_5'
intervention_size_list = [1, 1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../baselines/exps_of_result/enco/{exp_name}', exist_ok=True)

for idx, benchmark_name in enumerate(['01cancer', '02earthquake', '03survey', '04asia', '05sachs',  '06child', '07insurance', '08water', '09mildew', '10alarm', '11barley', '12hailfinder', '13hepar2', '14win95pts', '15pathfinder']):
    if idx==0 or idx>=8:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    
    know_pred_graph_path = f'../exps_of_result/enco/{exp_name}/{benchmark_name}_aug_graph.txt'

    with open(raw_dataset_path, 'rb') as f:
        data_list = pickle.load(f)
    with open(int_targets_path, 'rb') as f:
        know_targets_list = pickle.load(f)
    
    data_obs = np.array(data_list[0], dtype=np.int32)
    data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
    intervention_targets = know_targets_list[1:]
    all_nodes = [idx for idx in range(data_obs.shape[1])]
    adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
        
    all_nodes_index = [all_nodes.index(item) for item in all_nodes]
    intervention_targets_index = [all_nodes.index(sublist[0]) for sublist in intervention_targets]  # only support single intervention
    
    real_exclude_inters = np.array(list(set(all_nodes_index) - set(intervention_targets_index)), dtype=np.int32)
    
    modify_data_int = np.ones((len(all_nodes), data_int.shape[1], data_int.shape[2]), dtype=np.int32)
    for i in range(data_int.shape[0]):
        modify_data_int[intervention_targets_index[i]] = data_int[i]
    
    modify_graph = CausalDAGDataset(adj_matrix=adj_matrix, data_obs=data_obs, data_int=modify_data_int, exclude_inters=real_exclude_inters)
    
    lambda_sparse = 0.02 if len(all_nodes) > 100 else 0.002
    # num_epochs = 50 if len(all_nodes) > 100 else 100
    num_epochs = 40 if len(all_nodes) > 100 else 20
    
    try:
        # with HiddenPrints():
        discovery_module = ENCO(graph=modify_graph, sample_size_obs=len(data_obs), sample_size_inters=data_int.shape[1], lambda_sparse=lambda_sparse)
        discovery_module.to(torch.device('cuda:1'))

        pred_I_CPDAG_know = discovery_module.discover_graph(num_epochs=num_epochs).detach().numpy().astype(np.uint8)

        with open(know_pred_graph_path, 'wb') as f:
            np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
                
        pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
        mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
        mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
        print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')

    except:
        print(f'pass {benchmark_name}\n')

In [ ]:


for exp_name in  ['exp_3_1', 'exp_3_2', 'exp_3_3', 'exp_3_4', 'exp_3_5']:
    os.makedirs(f'../../baselines/exps_of_result/enco/{exp_name}', exist_ok=True)
    for idx, benchmark_name in enumerate(['01cancer', '02earthquake', '03survey', '04asia', '05sachs',  '06child', '07insurance', '08water', '09mildew', '10alarm', '11barley', '12hailfinder', '13hepar2', '14win95pts', '15pathfinder']):
        if idx!=5:
            continue
        print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
        if exp_name == 'exp_3_1':
            intervention_size = 0
        elif exp_name == 'exp_3_2':
            intervention_size = 8
        elif exp_name == 'exp_3_3':
            intervention_size = 12
        elif exp_name == 'exp_3_4':
            intervention_size = 16
        elif exp_name == 'exp_3_5':
            intervention_size = 20
        # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
        aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
        raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
        int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
        targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
        
        
        know_pred_graph_path = f'../exps_of_result/enco/{exp_name}/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        data_obs = np.array(data_list[0], dtype=np.int32)
        data_int = np.array([data.tolist() for data in data_list[1:]], dtype=np.int32)
        intervention_targets = know_targets_list[1:]
        all_nodes = [idx for idx in range(data_obs.shape[1])]
        adj_matrix = np.loadtxt(aug_graph_path, dtype=np.int32)[:-len(intervention_targets), :-len(intervention_targets)]
            
        all_nodes_index = [all_nodes.index(item) for item in all_nodes]
        intervention_targets_index = [all_nodes.index(sublist[0]) for sublist in intervention_targets]  # only support single intervention
        
        real_exclude_inters = np.array(list(set(all_nodes_index) - set(intervention_targets_index)), dtype=np.int32)
        
        modify_data_int = np.ones((len(all_nodes), data_int.shape[1], data_int.shape[2]), dtype=np.int32)
        for i in range(data_int.shape[0]):
            modify_data_int[intervention_targets_index[i]] = data_int[i]
        
        modify_graph = CausalDAGDataset(adj_matrix=adj_matrix, data_obs=data_obs, data_int=modify_data_int, exclude_inters=real_exclude_inters)
        
        lambda_sparse = 0.02 if len(all_nodes) > 100 else 0.002
        # num_epochs = 50 if len(all_nodes) > 100 else 100
        num_epochs = 40 if len(all_nodes) > 100 else 20
        
        try:
            # with HiddenPrints():
            discovery_module = ENCO(graph=modify_graph, sample_size_obs=len(data_obs), sample_size_inters=data_int.shape[1], lambda_sparse=lambda_sparse)
            discovery_module.to(torch.device('cuda:1'))

            pred_I_CPDAG_know = discovery_module.discover_graph(num_epochs=num_epochs).detach().numpy().astype(np.uint8)

            with open(know_pred_graph_path, 'wb') as f:
                np.savetxt(f, pred_I_CPDAG_know, fmt='%i')
                    
            pred_I_SKELETON_know = get_skeleton(pred_I_CPDAG_know)
            mt_skeleton_know = metric_skeleton_level(pred_I_SKELETON_know, targ_I_SKELETON)
            mt_cpdag_know = metric_cpdag_level(pred_I_CPDAG_know, targ_I_CPDAG)
            print(f'know performance: \n skeleton:{mt_skeleton_know} \n  cpdag:{mt_cpdag_know} \n')

        except:
            print(f'pass {benchmark_name}\n')